# AI-NIDS Dashboard

Multi-tier intrusion detection across NSL-KDD and UNSW-NB15. The five tiers (classical, boosting, deep learning, hybrid, stacking) plus SHAP/LIME explainability are loaded from the saved `results/metrics.json`. Re-run `python -m src.train --dataset <ds> --tier <t>` to refresh any cell.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

from src.config import CONFIG
from src.data.loader import load_nslkdd, load_unsw
from src.evaluate import all_results

sns.set_theme(style='whitegrid')
print('cwd:', os.getcwd())
print('metrics file exists:', CONFIG.paths.metrics_json.exists())

cwd: C:\Users\harsh\ai-nids
metrics file exists: True


## 1. Dataset Overview

NSL-KDD is the cleaned 1999 KDD-Cup successor; UNSW-NB15 (2015) is more recent and larger. Together they let us compare a 5-class IDS task (NSL-KDD) against a 10-class one (UNSW).

In [2]:
nsl_tr, nsl_y_tr, nsl_te, nsl_y_te, nsl_meta = load_nslkdd()
unsw_tr, unsw_y_tr, unsw_te, unsw_y_te, unsw_meta = load_unsw()

def _counts(y, meta):
    s = pd.Series(y).map(lambda i: meta.labels[i]).value_counts()
    return s.reindex(meta.labels).fillna(0).astype(int)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for row, (name, tr_y, te_y, meta) in enumerate([
    ('NSL-KDD', nsl_y_tr, nsl_y_te, nsl_meta),
    ('UNSW-NB15', unsw_y_tr, unsw_y_te, unsw_meta),
]):
    for col, (label, y) in enumerate([('train', tr_y), ('test', te_y)]):
        ax = axes[row, col]
        c = _counts(y, meta)
        ax.barh(range(len(c)), c.values, color=sns.color_palette('mako', len(c)))
        ax.set_yticks(range(len(c)))
        ax.set_yticklabels(c.index, fontsize=9)
        ax.set_title(f'{name} - {label} ({c.sum():,} rows)')
        ax.set_xlabel('count')
        for i, v in enumerate(c.values):
            ax.text(v, i, f'  {v:,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(CONFIG.paths.plots / 'class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Preprocessing Walkthrough

- Three categorical features per dataset (NSL-KDD: `protocol_type`, `service`, `flag`; UNSW: `proto`, `service`, `state`).
- `LabelEncoder` is fit on the training split only. Unseen test categories map to a fixed sentinel value to avoid leakage.
- `StandardScaler` is also fit on the training split only.
- For CatBoost, the raw dataframe is passed through with `cat_features` so the model handles categoricals natively.
- Difficulty column is dropped from NSL-KDD (it leaks how hard each row is).
- IDs and the redundant binary `label` are dropped from UNSW (the multi-class `attack_cat` is the source of truth).

In [3]:
from src.data.preprocess import fit_split
for name, loader in [('NSL-KDD', load_nslkdd), ('UNSW-NB15', load_unsw)]:
    Xtr, ytr, Xte, yte, meta = loader()
    split = fit_split(Xtr, ytr, Xte, yte, meta)
    print(f'{name:10s}  X_train={split.X_train.shape}  X_test={split.X_test.shape}  '
          f'cat={meta.categorical_cols}  classes={len(meta.labels)}')

NSL-KDD     X_train=(125973, 41)  X_test=(22544, 41)  cat=['protocol_type', 'service', 'flag']  classes=5


UNSW-NB15   X_train=(82332, 42)  X_test=(175341, 42)  cat=['proto', 'service', 'state']  classes=10


## 3. Final Comparison Table

All tiers, both datasets. Latency is measured over 1,000 records on the host CPU. Model size is the pickled estimator on disk.

In [4]:
store = all_results()
rows = []
for ds, bucket in store.items():
    for model_name, m in bucket.items():
        rows.append({
            'Dataset': ds,
            'Model': model_name,
            'Binary Acc': f"{m['binary_accuracy']*100:.2f}%",
            'Multi Acc': f"{m['multi_accuracy']*100:.2f}%",
            'Macro F1': f"{m['macro_f1']:.3f}",
            'Weighted F1': f"{m['weighted_f1']:.3f}",
            'ROC-AUC': f"{m['roc_auc']:.3f}" if m['roc_auc'] is not None else 'n/a',
            'PR-AUC': f"{m['pr_auc']:.3f}" if m['pr_auc'] is not None else 'n/a',
            'Inference (ms)': f"{m['inference_ms_per_record']:.3f}",
            'Train (s)': f"{m['train_time_s']:.1f}",
            'Size (MB)': f"{m['model_size_mb']:.1f}",
        })
df = pd.DataFrame(rows)
df

,Dataset,Model,Binary Acc,Multi Acc,Macro F1,Weighted F1,ROC-AUC,PR-AUC,Inference (ms),Train (s),Size (MB)
0,nslkdd,Random Forest,76.00%,75.03%,0.502,0.704,0.964,0.966,0.051,87.8,32.3
1,nslkdd,SVM (RBF),76.23%,74.66%,0.465,0.696,0.948,0.961,0.137,161.7,0.9
2,nslkdd,KNN,76.01%,74.37%,0.543,0.699,0.804,0.824,0.065,40.6,40.4
3,nslkdd,Voting Ensemble,75.54%,74.06%,0.480,0.692,0.969,0.973,0.315,119.9,147.1
4,nslkdd,XGBoost,78.66%,77.31%,0.551,0.734,0.970,0.972,0.003,91.6,2.2
5,nslkdd,LightGBM,77.13%,75.68%,0.590,0.718,0.973,0.973,0.018,125.1,14.9
6,nslkdd,CatBoost,77.42%,76.34%,0.533,0.723,0.969,0.971,0.004,304.7,5.0
7,nslkdd,1D-CNN,76.49%,74.95%,0.494,0.701,0.937,0.951,0.012,61.8,0.1
8,nslkdd,LSTM,77.30%,75.77%,0.489,0.712,0.938,0.945,0.013,50.7,0.2
9,nslkdd,Autoencoder,85.93%,71.15%,0.321,0.617,0.946,0.942,0.006,1.9,0.0


## 4. Per-dataset comparison charts

In [5]:
from src.evaluate import plot_model_comparison
for ds in ('nslkdd', 'unsw'):
    p = plot_model_comparison(ds)
    if p is None:
        continue
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(mpimg.imread(p)); ax.axis('off')
    plt.tight_layout(); plt.show()

## 5. Confusion matrices

Multi-class confusion matrices for the strongest model on each dataset. The full set is in `results/confusion_matrices/`.

In [6]:
for ds in ('nslkdd', 'unsw'):
    bucket = store.get(ds, {})
    if not bucket:
        continue
    best = max(bucket.items(), key=lambda kv: kv[1].get('macro_f1', 0))
    base = best[0].lower().replace(' ', '_').replace('(', '').replace(')', '')
    p_multi = CONFIG.paths.confusion_matrices / f'{ds}_{base}_multi.png'
    p_bin = CONFIG.paths.confusion_matrices / f'{ds}_{base}_binary.png'
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, p in zip(axes, [p_multi, p_bin], strict=True):
        if p.exists():
            ax.imshow(mpimg.imread(p))
        ax.axis('off')
    fig.suptitle(f'{ds}: best model = {best[0]}')
    plt.tight_layout(); plt.show()

## 6. SHAP global importance (Tier 2)

TreeExplainer attributions for the tuned XGBoost model. The beeswarm plot below shows how each top feature pushes the prediction toward (right) or away from (left) the dominant attack class. Rendered from `results/shap/`.

In [7]:
for ds in ('nslkdd', 'unsw'):
    for kind in ('global_bar', 'beeswarm', 'per_class'):
        p = CONFIG.paths.shap / f'{ds}_xgboost_{kind}.png'
        if not p.exists():
            continue
        fig, ax = plt.subplots(figsize=(9, 5.5))
        ax.imshow(mpimg.imread(p)); ax.axis('off')
        ax.set_title(f'{ds.upper()} - {kind.replace("_", " ")}')
        plt.tight_layout(); plt.show()

## 7. SHAP walkthrough on a single attack record

The waterfall plot below decomposes one model prediction. Bars push the log-odds up (red, toward attack class) or down (blue, toward another class). The base value is the model's average log-odds across the training set. The interpretation is below the plot.

In [8]:
from pathlib import Path
p = CONFIG.paths.shap / 'unsw_xgboost_waterfall.png'
if p.exists():
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(mpimg.imread(p)); ax.axis('off')
    ax.set_title('UNSW-NB15: waterfall on one attack record (XGBoost)')
    plt.tight_layout(); plt.show()
else:
    print('Run: python -m src.explain.shap_analysis to generate the waterfall plot.')

Run: python -m src.explain.shap_analysis to generate the waterfall plot.


## 8. LIME per-instance examples (Tier 3 deep model)

In [9]:
import os
files = sorted(CONFIG.paths.lime.glob('*.png'))
for fp in files[:4]:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.imshow(mpimg.imread(fp)); ax.axis('off')
    ax.set_title(fp.stem)
    plt.tight_layout(); plt.show()
if not files:
    print('No LIME plots found - run python -m src.explain.lime_analysis')

## 9. Honest takeaways

Bullet points are populated by hand in the README. Refer to the `Notes from the run` section there for what was actually surprising during this build.